# Trace, Charge Variance, And Flattened Energy Vs Cycle Protocol Comparison

Runs canonical GPU Markov dynamics for the `N20x40` domain-wall system and saves cycle-resolved per-sample trajectories for `real(trace(G))` and the flattened-H energy `Tr(H^T C)` with `C = (G + I)/2`.

For `maxmix` runs, the notebook also saves the full-system charge-variance trajectory `Tr[C(I-C)] = (Nlayer - Tr(G^2)) / 4`.

All observables are computed online with `cycle_observer`, so the notebook does not save covariance histories.


## Configurations And Domain-Wall Geometry

This notebook generates one fixed `N20x40_C50_dwtrunc1` campaign with the following settings:

| Nx | Ny | nshell | init modes | protocols / samples | cycles | dtype | dw_truncation | trial orbital | alpha_1 / alpha_2 |
|---:|---:|---:|---|---|---:|---|---|---|---|
| 20 | 40 | `1` | `maxmix`, `default` | `perfect_correction` / 10; `postselect` / 1 | 50 | `complex128` | `True` | `X` | 1 / 30 |

Saved observables:
- every-sample `Re tr(G)` trajectory for every run
- every-sample flattened-H energy trajectory `Tr(H^T C)` for every run
- every-sample full-system charge-variance trajectory for `maxmix` runs, including postselection

Domain-wall convention for this system size: `Nx=20` gives `half=Nx//2=10` and `w=floor(0.2*Nx)=4`, so the code stores `DW_loc=[5, 13]`. The topological `alpha_1` slab is `x=5..13` inclusive, the trivial `alpha_2` regions are `x=0..4` and `x=14..19`, and the domain-wall positions are reported at `x=5` and `x=13`.

The canonical GPU engine forces `postselect=True` to one deterministic sample, so this notebook records `postselect` with `samples=1` and `perfect_correction` with `samples=10`.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
import sys

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=False)

BUNDLE_NAME = 'colab_small_system_testing'

def is_bundle_root(path: Path) -> bool:
    return (path / 'src' / 'classA_U1FGTN_gpu.py').exists()

candidates = []
for base in [Path.cwd(), *Path.cwd().parents]:
    candidates.extend([base, base / BUNDLE_NAME])
if IN_COLAB:
    candidates.extend([
        Path('/content/drive/MyDrive') / BUNDLE_NAME,
        Path('/content/drive/MyDrive/class_A_fermionic_adaptive_circuit') / BUNDLE_NAME,
    ])

seen = set()
BUNDLE_ROOT = None
for candidate in candidates:
    candidate = candidate.resolve()
    if candidate in seen:
        continue
    seen.add(candidate)
    if is_bundle_root(candidate):
        BUNDLE_ROOT = candidate
        break
if BUNDLE_ROOT is None:
    raise FileNotFoundError(f'Could not locate {BUNDLE_NAME} with src/classA_U1FGTN_gpu.py')

SRC_DIR = BUNDLE_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f'[setup] bundle root: {BUNDLE_ROOT}')


In [ ]:
import gc
import os
from typing import Any

import numpy as np
from tqdm.auto import tqdm
import torch

from classA_U1FGTN_gpu import classA_U1FGTN_gpu

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is required for this Colab GPU campaign.')

torch.set_grad_enabled(False)
total_gib = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)

print(f'[setup] torch={torch.__version__}')
print(f'[setup] cuda device: {torch.cuda.get_device_name(0)}')
print(f'[setup] total gpu memory: {total_gib:.2f} GiB')
if total_gib < 35.0:
    print('[warn] This notebook is sized for an A100 40GB runtime.')


In [ ]:
NX = 20
NY = 40
NSHELL_VALUES = [1]
INIT_MODES = ['maxmix', 'default']
PERFECT_CORRECTION_SAMPLES = 10
POSTSELECT_SAMPLES = 1
CYCLES = 50
ALPHA_1 = 1.0
ALPHA_2 = 30.0
DW_TRUNCATION = True
DTYPE = 'complex128'
DEVICE = 'cuda:0'
BACKEND = 'local'
SEQUENCE = 'raster_y'
BATCH_SIZE = None
TRACE_EPS = 1e-12
ENERGY_IMAG_TOL = 1e-9
CHARGE_VARIANCE_IMAG_TOL = 1e-9
TRIAL_ORBITALS = 'X'
OBSERVABLES_FILE_NAME = 'observables_vs_cycle.npz'

PROTOCOL_CONFIGS = [
    {
        'protocol': 'perfect_correction',
        'postselect': False,
        'perfect_correction': True,
        'samples': PERFECT_CORRECTION_SAMPLES,
    },
    {
        'protocol': 'postselect',
        'postselect': True,
        'perfect_correction': False,
        'samples': POSTSELECT_SAMPLES,
    },
]

GPU_DATA_ROOT = BUNDLE_ROOT / 'gpu_data'
CAMPAIGN_NAME = 'trace_chargevar_flattened_energy_vs_cycle_protocols'
OUTPUT_ROOT = GPU_DATA_ROOT / CAMPAIGN_NAME
CAMPAIGN_ID = f'N{NX}x{NY}_C{CYCLES}_{"dwtrunc1" if DW_TRUNCATION else "dwtrunc0"}'
CAMPAIGN_ROOT = OUTPUT_ROOT / 'campaigns' / CAMPAIGN_ID
RUNS_ROOT = CAMPAIGN_ROOT / 'runs'
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

CONFIGS = [
    {
        'Nx': NX,
        'Ny': NY,
        'nshell': int(nshell),
        'init_mode': init_mode,
        **protocol_cfg,
    }
    for nshell in NSHELL_VALUES
    for init_mode in INIT_MODES
    for protocol_cfg in PROTOCOL_CONFIGS
]

CANONICAL_ENTRY_POINT = 'classA_U1FGTN_gpu.run_markov_circuit'
FLATTENED_H_DEFINITION = 'H_flat = P_Ap + P_Bp - P_Am - P_Bm, Hermitized, built from OW projectors for trial orbital X using the run nshell.'
FLATTENED_ENERGY_DEFINITION = 'Tr(H^T C) with C = (G + I) / 2.'
CHARGE_VARIANCE_DEFINITION = 'Tr[C(I-C)] = (Nlayer - Tr(G^2)) / 4 for the full top layer.'

print(json.dumps({
    'campaign_id': CAMPAIGN_ID,
    'campaign_name': CAMPAIGN_NAME,
    'configs': CONFIGS,
    'cycles': CYCLES,
    'saved_cycles': list(range(CYCLES + 1)),
    'init_modes': INIT_MODES,
    'protocols': PROTOCOL_CONFIGS,
    'dw_truncation': DW_TRUNCATION,
    'dtype': DTYPE,
    'trial_orbitals': TRIAL_ORBITALS,
    'canonical_dynamics_entry_point': CANONICAL_ENTRY_POINT,
    'flattened_h_definition': FLATTENED_H_DEFINITION,
    'flattened_energy_definition': FLATTENED_ENERGY_DEFINITION,
    'charge_variance_definition': CHARGE_VARIANCE_DEFINITION,
    'output_root': str(OUTPUT_ROOT),
    'campaign_root': str(CAMPAIGN_ROOT),
    'runs_root': str(RUNS_ROOT),
    'note': 'postselect=True uses one deterministic trajectory because the GPU driver forces samples=1.',
}, indent=2))


In [ ]:
def write_json_atomic(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_suffix(path.suffix + '.tmp')
    with tmp_path.open('w', encoding='utf-8') as fh:
        json.dump(payload, fh, indent=2, sort_keys=True)
    tmp_path.replace(path)


def save_npz_atomic(path: Path, **arrays: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_suffix(path.suffix + '.tmp')
    with tmp_path.open('wb') as fh:
        np.savez_compressed(fh, **arrays)
    tmp_path.replace(path)


def rel_to_gpu_data(path) -> str | None:
    if path is None:
        return None
    try:
        return str(Path(path).resolve().relative_to(GPU_DATA_ROOT.resolve()))
    except Exception:
        return None


def update_gpu_data_index() -> None:
    index_path = GPU_DATA_ROOT / 'index.json'
    if index_path.exists():
        with index_path.open('r', encoding='utf-8') as fh:
            payload = json.load(fh)
    else:
        payload = {'bundle_root': str(BUNDLE_ROOT), 'gpu_data_root': str(GPU_DATA_ROOT), 'campaigns': []}
    campaigns = payload.setdefault('campaigns', [])
    entry = {
        'name': CAMPAIGN_NAME,
        'kind': 'derived_trajectory_observables',
        'manifest': f'{CAMPAIGN_NAME}/latest_campaign.json',
    }
    campaigns[:] = [item for item in campaigns if item.get('name') != CAMPAIGN_NAME]
    campaigns.append(entry)
    payload['bundle_root'] = str(BUNDLE_ROOT)
    payload['gpu_data_root'] = str(GPU_DATA_ROOT)
    write_json_atomic(index_path, payload)


def case_output_dir(cfg: dict) -> Path:
    return RUNS_ROOT / f"nsh{cfg['nshell']}_init-{cfg['init_mode']}_{cfg['protocol']}"


def expected_samples_for_config(cfg: dict) -> int:
    return 1 if bool(cfg['postselect']) else int(cfg['samples'])


def validate_top_layer_batch(G_batch: torch.Tensor) -> None:
    if G_batch.ndim != 3:
        raise ValueError(f'Expected G_batch with shape (B,N,N), got {tuple(G_batch.shape)}')
    batch, nlayer, nlayer_2 = G_batch.shape
    expected = 2 * int(NX) * int(NY)
    if nlayer != nlayer_2 or nlayer != expected:
        raise ValueError(f'Expected covariance shape (B,{expected},{expected}), got {tuple(G_batch.shape)}')
    if not torch.isfinite(G_batch).all():
        bad = torch.nonzero(~torch.isfinite(G_batch), as_tuple=False)[0].detach().cpu().tolist()
        raise FloatingPointError(f'Non-finite covariance batch entry encountered at index {bad}')


def trace_real_batch_torch(G_batch: torch.Tensor) -> torch.Tensor:
    validate_top_layer_batch(G_batch)
    return torch.diagonal(G_batch, dim1=-2, dim2=-1).sum(dim=-1).real


def build_flattened_hamiltonian_torch(model: classA_U1FGTN_gpu, *, nshell: int, trial_orbitals: str, dw_truncation: bool) -> dict[str, Any]:
    trial_orbitals = str(trial_orbitals).upper()
    model.construct_OW_projectors(
        nshell=int(nshell),
        DW=True,
        trial_orbitals=trial_orbitals,
        dw_truncation=bool(dw_truncation),
    )
    nlayer = int(model.Nlayer)
    WA_p = model.WF_Ap.reshape(nlayer, -1)
    WB_p = model.WF_Bp.reshape(nlayer, -1)
    WA_m = model.WF_Am.reshape(nlayer, -1)
    WB_m = model.WF_Bm.reshape(nlayer, -1)

    P_Ap = WA_p @ WA_p.conj().T
    P_Bp = WB_p @ WB_p.conj().T
    P_Am = WA_m @ WA_m.conj().T
    P_Bm = WB_m @ WB_m.conj().T

    H_flat = P_Ap + P_Bp - P_Am - P_Bm
    H_flat = 0.5 * (H_flat + H_flat.conj().T)
    herm_err = float(torch.max(torch.abs(H_flat - H_flat.conj().T)).detach().cpu().item())
    trace_H = torch.trace(H_flat)
    return {
        'H_flat': H_flat,
        'trace_H': trace_H,
        'trace_H_real': float(trace_H.real.detach().cpu().item()),
        'trace_H_imag': float(trace_H.imag.detach().cpu().item()),
        'hermiticity_error': herm_err,
        'trial_orbitals': trial_orbitals,
        'nshell': int(nshell),
    }


def flattened_energy_batch_torch(*, G_batch: torch.Tensor, H_flat: torch.Tensor, trace_H: torch.Tensor) -> torch.Tensor:
    validate_top_layer_batch(G_batch)
    pair_term = torch.einsum('ij,bij->b', H_flat, G_batch)
    energy_vals = 0.5 * (pair_term + trace_H)
    if not torch.isfinite(energy_vals).all():
        bad = torch.nonzero(~torch.isfinite(energy_vals), as_tuple=False)[0].detach().cpu().tolist()
        raise FloatingPointError(f'Non-finite flattened energy encountered at batch entry {bad}')
    return energy_vals


def charge_variance_batch_torch(*, G_batch: torch.Tensor, nlayer: int) -> torch.Tensor:
    validate_top_layer_batch(G_batch)
    tr_G_sq = torch.einsum('bij,bji->b', G_batch, G_batch)
    variance_vals = 0.25 * (float(nlayer) - tr_G_sq)
    if not torch.isfinite(variance_vals).all():
        bad = torch.nonzero(~torch.isfinite(variance_vals), as_tuple=False)[0].detach().cpu().tolist()
        raise FloatingPointError(f'Non-finite charge variance encountered at batch entry {bad}')
    return variance_vals


def summarize_sample_trajectories(values: np.ndarray) -> dict[str, np.ndarray]:
    values = np.asarray(values, dtype=np.float64)
    if values.ndim != 2:
        raise ValueError(f'Expected (samples, cycles) array, got shape {values.shape}')
    if not np.isfinite(values).all():
        bad = np.argwhere(~np.isfinite(values))[0].tolist()
        raise FloatingPointError(f'Non-finite saved trajectory entry at index {bad}')
    sample_count, time_count = values.shape
    sums = np.sum(values, axis=0, dtype=np.float64)
    sumsq = np.sum(values * values, axis=0, dtype=np.float64)
    counts = np.full((time_count,), sample_count, dtype=np.int64)
    mean = sums / float(sample_count)
    std = np.full((time_count,), np.nan, dtype=np.float64)
    stderr = np.full((time_count,), np.nan, dtype=np.float64)
    if sample_count > 1:
        numer = sumsq - (sums * sums) / float(sample_count)
        numer = np.maximum(numer, 0.0)
        var = numer / float(sample_count - 1)
        std = np.sqrt(var)
        stderr = std / np.sqrt(float(sample_count))
    return {
        'cycles': np.arange(time_count, dtype=np.int64),
        'sample_counts_by_cycle': counts,
        'sum': sums,
        'sumsq': sumsq,
        'mean': mean,
        'std': std,
        'stderr': stderr,
    }


In [ ]:
def run_one_config(cfg: dict) -> dict:
    out_dir = case_output_dir(cfg)
    out_dir.mkdir(parents=True, exist_ok=True)
    run_samples_requested = int(cfg['samples'])
    expected_samples = expected_samples_for_config(cfg)
    protocol = str(cfg['protocol'])
    init_mode = str(cfg['init_mode'])
    postselect = bool(cfg['postselect'])
    perfect_correction = bool(cfg['perfect_correction'])
    save_charge_variance = init_mode == 'maxmix'
    print(
        f"[config] Nx={cfg['Nx']} Ny={cfg['Ny']} nshell={cfg['nshell']} "
        f"init={init_mode} protocol={protocol} cycles={CYCLES} requested_samples={run_samples_requested} output={out_dir}"
    )

    model = classA_U1FGTN_gpu(
        Nx=int(cfg['Nx']),
        Ny=int(cfg['Ny']),
        DW=True,
        nshell=int(cfg['nshell']),
        filling_frac=0.5,
        alpha_1=ALPHA_1,
        alpha_2=ALPHA_2,
        trial_orbitals=TRIAL_ORBITALS,
        dw_truncation=DW_TRUNCATION,
        triv_region_local_mode=False,
        device=DEVICE,
        dtype=DTYPE,
        backend=BACKEND,
    )

    flattened_h_data = build_flattened_hamiltonian_torch(
        model,
        nshell=int(cfg['nshell']),
        trial_orbitals=TRIAL_ORBITALS,
        dw_truncation=DW_TRUNCATION,
    )

    trace_real_by_sample = np.full((expected_samples, CYCLES + 1), np.nan, dtype=np.float64)
    flattened_energy_by_sample = np.full((expected_samples, CYCLES + 1), np.nan, dtype=np.float64)
    charge_variance_by_sample = (
        np.full((expected_samples, CYCLES + 1), np.nan, dtype=np.float64)
        if save_charge_variance
        else None
    )
    observer_stats = {
        'flattened_energy_imag_max_abs': 0.0,
        'charge_variance_imag_max_abs': 0.0,
    }

    observer_pbar = tqdm(
        total=expected_samples * (CYCLES + 1),
        desc='Observable observer',
        unit='sample-cycle',
        leave=False,
    )

    def observable_observer(*, cycle, G, batch_index, batch_start, batch_count):
        cycle = int(cycle)
        batch_start = int(batch_start)
        batch_count = int(batch_count)
        sl = slice(batch_start, batch_start + batch_count)

        trace_vals = trace_real_batch_torch(G)
        trace_real_by_sample[sl, cycle] = trace_vals.detach().cpu().numpy().astype(np.float64, copy=False)

        flattened_energy_vals = flattened_energy_batch_torch(
            G_batch=G,
            H_flat=flattened_h_data['H_flat'],
            trace_H=flattened_h_data['trace_H'],
        )
        flattened_energy_imag_max = float(torch.max(torch.abs(flattened_energy_vals.imag)).detach().cpu().item())
        observer_stats['flattened_energy_imag_max_abs'] = max(
            observer_stats['flattened_energy_imag_max_abs'],
            flattened_energy_imag_max,
        )
        if flattened_energy_imag_max > ENERGY_IMAG_TOL:
            raise FloatingPointError(
                f'Flattened energy imaginary part exceeded tolerance at cycle={cycle}: '
                f'{flattened_energy_imag_max:.6e} > {ENERGY_IMAG_TOL:.6e}'
            )
        flattened_energy_by_sample[sl, cycle] = flattened_energy_vals.real.detach().cpu().numpy().astype(np.float64, copy=False)

        if charge_variance_by_sample is not None:
            charge_variance_vals = charge_variance_batch_torch(G_batch=G, nlayer=int(model.Nlayer))
            charge_variance_imag_max = float(torch.max(torch.abs(charge_variance_vals.imag)).detach().cpu().item())
            observer_stats['charge_variance_imag_max_abs'] = max(
                observer_stats['charge_variance_imag_max_abs'],
                charge_variance_imag_max,
            )
            if charge_variance_imag_max > CHARGE_VARIANCE_IMAG_TOL:
                raise FloatingPointError(
                    f'Charge variance imaginary part exceeded tolerance at cycle={cycle}: '
                    f'{charge_variance_imag_max:.6e} > {CHARGE_VARIANCE_IMAG_TOL:.6e}'
                )
            charge_variance_by_sample[sl, cycle] = charge_variance_vals.real.detach().cpu().numpy().astype(np.float64, copy=False)

        observer_pbar.update(batch_count)

    try:
        result = model.run_markov_circuit(
            G_history=False,
            progress=False,
            cycles=CYCLES,
            postselect=postselect,
            perfect_correction=perfect_correction,
            samples=run_samples_requested,
            init_mode=init_mode,
            save=False,
            save_init=False,
            save_history_stride=None,
            n_a=0.5,
            sequence=SEQUENCE,
            batch_size=BATCH_SIZE,
            return_data=False,
            cycle_observer=observable_observer,
        )
    finally:
        observer_pbar.close()

    actual_samples = int(result['samples'])
    if actual_samples != expected_samples:
        raise RuntimeError(f'Expected {expected_samples} samples but run returned {actual_samples}.')

    if np.isnan(trace_real_by_sample).any():
        raise RuntimeError('Trace trajectory array was not filled at every cycle/sample entry.')
    if np.isnan(flattened_energy_by_sample).any():
        raise RuntimeError('Flattened-energy trajectory array was not filled at every cycle/sample entry.')
    if charge_variance_by_sample is not None and np.isnan(charge_variance_by_sample).any():
        raise RuntimeError('Charge-variance trajectory array was not filled at every cycle/sample entry.')

    trace_data = summarize_sample_trajectories(trace_real_by_sample)
    flattened_energy_data = summarize_sample_trajectories(flattened_energy_by_sample)
    charge_variance_data = summarize_sample_trajectories(charge_variance_by_sample) if charge_variance_by_sample is not None else None

    if not np.all(trace_data['sample_counts_by_cycle'] == actual_samples):
        raise RuntimeError(
            f"Expected {actual_samples} trace observations per cycle, got {trace_data['sample_counts_by_cycle'].tolist()}"
        )
    if not np.all(flattened_energy_data['sample_counts_by_cycle'] == actual_samples):
        raise RuntimeError(
            'Flattened-energy sample counts do not match the actual sample count.'
        )
    if init_mode == 'maxmix':
        if np.max(np.abs(trace_real_by_sample[:, 0])) > 1e-9:
            raise RuntimeError(
                f"Expected maxmix initial trace to be near 0, got {trace_real_by_sample[:, 0].tolist()}"
            )
        if np.max(np.abs(flattened_energy_by_sample[:, 0])) > 1e-8:
            raise RuntimeError(
                f"Expected maxmix initial flattened energy to be near 0, got {flattened_energy_by_sample[:, 0].tolist()}"
            )
        expected_initial_charge_variance = 0.25 * float(model.Nlayer)
        if charge_variance_by_sample is None:
            raise RuntimeError('Expected charge-variance trajectories for maxmix runs.')
        if np.max(np.abs(charge_variance_by_sample[:, 0] - expected_initial_charge_variance)) > 1e-8:
            raise RuntimeError(
                'Expected maxmix initial charge variance to equal Nlayer / 4 within tolerance.'
            )

    batch_size = int(result['batch_size'])
    batch_size_mode = result['batch_size_mode']
    batch_size_auto_info = result.get('batch_size_auto_info')
    saved_observables = ['trace_real', 'flattened_energy']
    if charge_variance_data is not None:
        saved_observables.append('charge_variance')

    npz_path = out_dir / OBSERVABLES_FILE_NAME
    config_json = json.dumps({
        'Nx': int(cfg['Nx']),
        'Ny': int(cfg['Ny']),
        'nshell': int(cfg['nshell']),
        'cycles': CYCLES,
        'protocol': protocol,
        'postselect': postselect,
        'perfect_correction': perfect_correction,
        'samples_requested': run_samples_requested,
        'samples_actual': actual_samples,
        'init_mode': init_mode,
        'alpha_1': ALPHA_1,
        'alpha_2': ALPHA_2,
        'dw_truncation': DW_TRUNCATION,
        'dtype': DTYPE,
        'sequence': SEQUENCE,
        'trial_orbitals': TRIAL_ORBITALS,
        'canonical_dynamics_entry_point': CANONICAL_ENTRY_POINT,
        'flattened_h_definition': FLATTENED_H_DEFINITION,
        'flattened_h_nshell': int(flattened_h_data['nshell']),
        'flattened_energy_definition': FLATTENED_ENERGY_DEFINITION,
        'charge_variance_definition': CHARGE_VARIANCE_DEFINITION,
        'saved_observables': saved_observables,
        'charge_variance_saved': charge_variance_data is not None,
    }, sort_keys=True)

    save_kwargs = {
        'cycles': trace_data['cycles'],
        'sample_counts_by_cycle': trace_data['sample_counts_by_cycle'],
        'trace_real_by_sample': trace_real_by_sample,
        'trace_real_mean': trace_data['mean'],
        'trace_real_std': trace_data['std'],
        'trace_real_stderr': trace_data['stderr'],
        'trace_real_sum': trace_data['sum'],
        'trace_real_sumsq': trace_data['sumsq'],
        'flattened_energy_by_sample': flattened_energy_by_sample,
        'flattened_energy_mean': flattened_energy_data['mean'],
        'flattened_energy_std': flattened_energy_data['std'],
        'flattened_energy_stderr': flattened_energy_data['stderr'],
        'flattened_energy_sum': flattened_energy_data['sum'],
        'flattened_energy_sumsq': flattened_energy_data['sumsq'],
        'config_json': np.asarray(config_json),
    }
    if charge_variance_data is not None:
        save_kwargs.update({
            'charge_variance_by_sample': charge_variance_by_sample,
            'charge_variance_mean': charge_variance_data['mean'],
            'charge_variance_std': charge_variance_data['std'],
            'charge_variance_stderr': charge_variance_data['stderr'],
            'charge_variance_sum': charge_variance_data['sum'],
            'charge_variance_sumsq': charge_variance_data['sumsq'],
        })
    save_npz_atomic(npz_path, **save_kwargs)

    summary_path = out_dir / 'run_summary.json'
    summary = {
        'canonical_dynamics_entry_point': CANONICAL_ENTRY_POINT,
        'campaign_name': CAMPAIGN_NAME,
        'campaign_id': CAMPAIGN_ID,
        'campaign_root': str(CAMPAIGN_ROOT),
        'campaign_root_relative': rel_to_gpu_data(CAMPAIGN_ROOT),
        'run_dir': str(out_dir),
        'run_dir_relative': rel_to_gpu_data(out_dir),
        'npz_path': str(npz_path),
        'npz_path_relative': rel_to_gpu_data(npz_path),
        'saved_observable': 'per-sample trajectories for Re tr(G), flattened-H energy, and optional charge variance',
        'saved_observables': saved_observables,
        'covariance_history_saved': False,
        'observable_collection_mode': 'cycle_observer_streaming',
        'Nx': int(cfg['Nx']),
        'Ny': int(cfg['Ny']),
        'nshell': int(cfg['nshell']),
        'cycles': CYCLES,
        'saved_cycles': trace_data['cycles'].astype(int).tolist(),
        'protocol': protocol,
        'postselect': postselect,
        'perfect_correction': perfect_correction,
        'init_mode': init_mode,
        'samples_requested': run_samples_requested,
        'samples': actual_samples,
        'postselect_forces_single_sample': bool(postselect),
        'note': 'postselect=True uses one deterministic trajectory because the canonical GPU driver forces samples=1.' if postselect else None,
        'alpha_1': ALPHA_1,
        'alpha_2': ALPHA_2,
        'dw_truncation': DW_TRUNCATION,
        'dtype_requested': DTYPE,
        'dtype_resolved': DTYPE,
        'device': DEVICE,
        'backend_requested': BACKEND,
        'backend_resolved': str(model.backend),
        'sequence': SEQUENCE,
        'trial_orbitals': TRIAL_ORBITALS,
        'batch_size_requested': BATCH_SIZE,
        'batch_size': batch_size,
        'batch_size_mode': batch_size_mode,
        'batch_size_auto_info': batch_size_auto_info,
        'flattened_h_definition': FLATTENED_H_DEFINITION,
        'flattened_h_nshell': int(flattened_h_data['nshell']),
        'flattened_h_trial_orbitals': flattened_h_data['trial_orbitals'],
        'flattened_h_hermiticity_error': float(flattened_h_data['hermiticity_error']),
        'flattened_h_trace_real': float(flattened_h_data['trace_H_real']),
        'flattened_h_trace_imag': float(flattened_h_data['trace_H_imag']),
        'flattened_energy_definition': FLATTENED_ENERGY_DEFINITION,
        'flattened_energy_saved': True,
        'flattened_energy_imag_max_abs': float(observer_stats['flattened_energy_imag_max_abs']),
        'charge_variance_definition': CHARGE_VARIANCE_DEFINITION,
        'charge_variance_saved': charge_variance_data is not None,
        'charge_variance_imag_max_abs': None if charge_variance_data is None else float(observer_stats['charge_variance_imag_max_abs']),
        'trace_real_mean_initial': float(trace_data['mean'][0]),
        'trace_real_mean_final': float(trace_data['mean'][-1]),
        'trace_real_stderr_initial': None if np.isnan(trace_data['stderr'][0]) else float(trace_data['stderr'][0]),
        'trace_real_stderr_final': None if np.isnan(trace_data['stderr'][-1]) else float(trace_data['stderr'][-1]),
        'flattened_energy_mean_initial': float(flattened_energy_data['mean'][0]),
        'flattened_energy_mean_final': float(flattened_energy_data['mean'][-1]),
        'flattened_energy_stderr_initial': None if np.isnan(flattened_energy_data['stderr'][0]) else float(flattened_energy_data['stderr'][0]),
        'flattened_energy_stderr_final': None if np.isnan(flattened_energy_data['stderr'][-1]) else float(flattened_energy_data['stderr'][-1]),
        'charge_variance_mean_initial': None if charge_variance_data is None else float(charge_variance_data['mean'][0]),
        'charge_variance_mean_final': None if charge_variance_data is None else float(charge_variance_data['mean'][-1]),
        'charge_variance_stderr_initial': None if charge_variance_data is None or np.isnan(charge_variance_data['stderr'][0]) else float(charge_variance_data['stderr'][0]),
        'charge_variance_stderr_final': None if charge_variance_data is None or np.isnan(charge_variance_data['stderr'][-1]) else float(charge_variance_data['stderr'][-1]),
    }
    write_json_atomic(summary_path, summary)

    del model, result, flattened_h_data
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return summary


In [ ]:
all_results = []
for cfg in tqdm(CONFIGS, desc='Observable configs', unit='config'):
    all_results.append(run_one_config(cfg))

campaign_manifest = {
    'canonical_dynamics_entry_point': CANONICAL_ENTRY_POINT,
    'bundle_root': str(BUNDLE_ROOT),
    'gpu_data_root': str(GPU_DATA_ROOT),
    'output_root': str(OUTPUT_ROOT),
    'campaign_name': CAMPAIGN_NAME,
    'campaign_id': CAMPAIGN_ID,
    'campaign_root': str(CAMPAIGN_ROOT),
    'runs_root': str(RUNS_ROOT),
    'output_root_relative': rel_to_gpu_data(OUTPUT_ROOT),
    'campaign_root_relative': rel_to_gpu_data(CAMPAIGN_ROOT),
    'runs_root_relative': rel_to_gpu_data(RUNS_ROOT),
    'kind': 'derived_trajectory_observables',
    'configs': CONFIGS,
    'results': all_results,
    'cycles': CYCLES,
    'saved_cycles': list(range(CYCLES + 1)),
    'init_modes': INIT_MODES,
    'protocols': PROTOCOL_CONFIGS,
    'dw_truncation': DW_TRUNCATION,
    'dtype': DTYPE,
    'trial_orbitals': TRIAL_ORBITALS,
    'saved_observable': 'per-sample trajectories for Re tr(G), flattened-H energy, and optional charge variance',
    'saved_observables': [
        'trace_real_by_sample',
        'flattened_energy_by_sample',
        'charge_variance_by_sample for maxmix runs only',
    ],
    'charge_variance_saved_for_init_modes': ['maxmix'],
    'covariance_history_saved': False,
    'observable_collection_mode': 'cycle_observer_streaming',
    'flattened_h_definition': FLATTENED_H_DEFINITION,
    'flattened_energy_definition': FLATTENED_ENERGY_DEFINITION,
    'charge_variance_definition': CHARGE_VARIANCE_DEFINITION,
}
campaign_manifest_path = CAMPAIGN_ROOT / 'campaign_manifest.json'
write_json_atomic(campaign_manifest_path, campaign_manifest)

latest_campaign = {
    'name': CAMPAIGN_NAME,
    'kind': 'derived_trajectory_observables',
    'active_campaign_id': CAMPAIGN_ID,
    'campaign_root': rel_to_gpu_data(CAMPAIGN_ROOT),
    'manifest': rel_to_gpu_data(campaign_manifest_path),
}
write_json_atomic(OUTPUT_ROOT / 'latest_campaign.json', latest_campaign)
update_gpu_data_index()
print(f'[done] campaign manifest: {campaign_manifest_path}')
print(f'[done] latest campaign: {OUTPUT_ROOT / "latest_campaign.json"}')


In [ ]:
if IN_COLAB:
    from google.colab import runtime  # type: ignore
    runtime.unassign()
else:
    print('[done] not running in Colab; runtime disconnect skipped')
